In [1]:
import os
import json
from dataclasses import dataclass
from pydantic import BaseModel
from typing import Literal

from dotenv import load_dotenv

load_dotenv()


@dataclass(frozen=True)
class Provider:
    """One provider described as pure DATA (same design as Notebook 1)."""

    name: str
    env_var: str
    is_free: bool
    base_url: str | None
    model: str


PROVIDERS = [
    Provider("OpenAI",     "OPENAI_API_KEY",     False, None,                              "gpt-4o-mini"),
    Provider("Groq",       "GROQ_API_KEY",       True,  "https://api.groq.com/openai/v1", "llama-3.3-70b-versatile"),
]


def select_provider() -> Provider:
    for provider in PROVIDERS:
        if os.environ.get(provider.env_var):
            return provider
    expected = ", ".join(p.env_var for p in PROVIDERS)
    raise RuntimeError(f"No provider key set. Add one of {expected} to your .env file.")


def build_client(provider: Provider):
    from openai import OpenAI

    api_key = os.environ[provider.env_var]
    if provider.base_url is None:
        return OpenAI(api_key=api_key)
    return OpenAI(api_key=api_key, base_url=provider.base_url)


def have_any_key() -> bool:
    return any(os.environ.get(p.env_var) for p in PROVIDERS)



def llm_reply(prompt: str, *, max_tokens: int = 400) -> str:
    """Send one user prompt; return the assistant's text."""
    provider = select_provider()
    client = build_client(provider)
    result = client.chat.completions.create(
        model=provider.model,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    return result.choices[0].message.content

In [2]:
WEATHER_DB = {
    "london": {"celcius": 22, "sky": "cloudy"},
    "paris": {"celcius": 24, "sky": "sunny"},
    "tallinn": {"celcius": 28, "sky": "sunny"},
    "moscow": {"celcius": 18, "sky": "rainy"},
    "tokyo": {"celcius": 30, "sky": "sunny"},
    "beijing": {"celcius": 26, "sky": "cloudy"},
    "new york": {"celcius": 28, "sky": "sunny"},
    "mumbai": {"celcius": 32, "sky": "sunny"},
    "cape town": {"celcius": 16, "sky": "cloudy"},
    "sydney": {"celcius": 20, "sky": "sunny"},
    "rio de janeiro": {"celcius": 22, "sky": "cloudy"},
    "cairo": {"celcius": 24, "sky": "sunny"},
    "mexico city": {"celcius": 20, "sky": "cloudy"},
    
}

# Below is a tool defined manually by us, which can be called on our machine, and llm will just signal should we call this tool or not ?

In [3]:
def lookup_weather(location: str) -> str:
    """Look up the weather for a location."""
    record = WEATHER_DB.get(location.lower())
    if record is None:
        return f"Sorry, I don't know the weather in {location}."
    return f"The weather in {location} is {record['celcius']}C and {record['sky']}."

In [4]:
weather_schema = {
    "type": "function",
    "function": {
        "name": "lookup_weather",
        "description": "Look up the weather for a location.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "The location to look up the weather for."}
            },
            "required": ["location"]
        }
    }
} # this schema will be given to the llm

TOOLS = {
    "lookup_weather": lookup_weather,
    "stock_price_checker": None
}

In [5]:
def ask_llm_with_tool(prompt: str, *, max_tokens: int = 400) -> str:
    """
    Call the LLM with a tool call.
    """
    provider = select_provider()
    client = build_client(provider)
    result = client.chat.completions.create(
        model=provider.model,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
        tools=[weather_schema],
    )
    print(result.choices)
    return result.choices[0].message

In [6]:
msg = ask_llm_with_tool("What is the weather in Tokyo?")

print("final response from llm", msg)

[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_wLEQKCZF3Q4xTdXMEVmsupI8', function=Function(arguments='{"location":"Tokyo"}', name='lookup_weather'), type='function')]))]
final response from llm ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_wLEQKCZF3Q4xTdXMEVmsupI8', function=Function(arguments='{"location":"Tokyo"}', name='lookup_weather'), type='function')])


In [7]:
msg = ask_llm_with_tool("What is the weather in london compared to paris?")

print("final response from llm", msg)

[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_Cv7gjOs4yHGN63sUz8TOuDIC', function=Function(arguments='{"location": "London"}', name='lookup_weather'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_egO3lAPqYWWl2oQzhYrBtjKt', function=Function(arguments='{"location": "Paris"}', name='lookup_weather'), type='function')]))]
final response from llm ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_Cv7gjOs4yHGN63sUz8TOuDIC', function=Function(arguments='{"location": "London"}', name='lookup_weather'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_egO3lAPqYWWl2oQzhYrBtjKt', function=Function(arguments='{"location": "Paris"}', name='lookup_

In [8]:
if msg.tool_calls:
    print("tool call detected")
    print("Total tool calls", len(msg.tool_calls))
    for tool_call in msg.tool_calls:
        print("Tool call name", tool_call.function.name)
        print("Tool call arguments", tool_call.function.arguments)
        city = json.loads(tool_call.function.arguments)["location"]
        print(f"Calling tool for {city}")
        tool = TOOLS[tool_call.function.name] # this will give us the function 
        result = tool(city)
        print(result)

        print("-"*100)
else:
    print("No tool call detected")
    

tool call detected
Total tool calls 2
Tool call name lookup_weather
Tool call arguments {"location": "London"}
Calling tool for London
The weather in London is 22C and cloudy.
----------------------------------------------------------------------------------------------------
Tool call name lookup_weather
Tool call arguments {"location": "Paris"}
Calling tool for Paris
The weather in Paris is 24C and sunny.
----------------------------------------------------------------------------------------------------


In [9]:
def ask_llm_with_tool(prompt: str, *, max_tokens: int = 400) -> str:
    """
    Call the LLM with a tool call.
    """
    provider = select_provider()
    client = build_client(provider)
    messages = [
        {"role": "user", "content": prompt},
    ]

    while True:
        response = client.chat.completions.create(
            model=provider.model,
            max_tokens=max_tokens,
            messages=messages,
            tools=[weather_schema],
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content

        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": msg.tool_calls,
        })

        # llm is asking us to call a tool
        for tool_call in msg.tool_calls:
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments) # {"location": "london"}
            tool_call_id = tool_call.id

            if tool_name not in TOOLS:
                raise ValueError(f"Tool {tool_name} not found")

            tool = TOOLS[tool_name] # actual function
            result = tool(**tool_args) # -> lookup_weather(location="london")

            messages.append(
                {"role": "tool", "tool_call_id": tool_call_id, "content": result}
            )

        # print("All tool calls done, going back to LLM")


            

In [10]:
ask_llm_with_tool("What is the weather like in London compared to Paris?")

'The weather in London is 22°C and cloudy, while in Paris, it is 24°C and sunny.'